# A02 demonstration: TensorFlow vs. PyTorch
**Individual project • ITAI 2376**

Both frameworks learn to classify points from the **same synthetic two-feature dataset**. Compare the code structure, not training speed. **This notebook does not claim a controlled speed benchmark.**

The course Module 02 notebook demonstrates TensorFlow VGG16 image classification. This supplemental notebook makes it easier to show both frameworks side by side.

## Setup
Install `numpy`, `matplotlib`, `tensorflow`, and `torch` in a suitable Python/Colab environment if they are missing. CPU is sufficient for this small example. Note: TensorFlow is not preinstalled in every runtime.

In [1]:
import numpy as np
np.random.seed(42)
X = np.random.randn(600, 2).astype('float32')
y = ((X[:, 0]**2 + X[:, 1]**2) > 1.4).astype('float32').reshape(-1,1)
print('Examples:',len(X),'Positive class:',round(float(y.mean()),3))

Examples: 600 Positive class: 0.472


## TensorFlow (Keras): concise high-level API

In [2]:
import tensorflow as tf
tf.random.set_seed(42)
tf_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(2,)),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])
tf_model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
tf_model.fit(X,y,epochs=20,batch_size=32,verbose=0)
print('TensorFlow training accuracy:',round(tf_model.evaluate(X,y,verbose=0)[1],3))

TensorFlow training accuracy: 0.858


## PyTorch: explicit training loop

In [3]:
import torch
from torch import nn
torch.manual_seed(42)
model = nn.Sequential(nn.Linear(2,16),nn.ReLU(),nn.Linear(16,1))
opt = torch.optim.Adam(model.parameters(),lr=0.01)
loss_fn = nn.BCEWithLogitsLoss()
features = torch.from_numpy(X); targets = torch.from_numpy(y)
for epoch in range(20):
    for ids in torch.randperm(len(features)).split(32):
        opt.zero_grad()
        loss = loss_fn(model(features[ids]),targets[ids])
        loss.backward()
        opt.step()
with torch.no_grad():
    preds=(torch.sigmoid(model(features))>=0.5).float()
    print('PyTorch training accuracy:',round(float((preds==targets).float().mean()),3))

PyTorch training accuracy: 0.993


## Discussion prompts
1. Which example makes the training steps easier to see?

PyTorch makes the training steps easier to see because it uses an explicit training loop. We can observe how the model makes predictions, calculates the loss, computes gradients, and updates its parameters. This gives us a clearer understanding of how deep learning models are trained.



2. Which example uses fewer lines to fit a model?

TensorFlow, using the Keras API, requires fewer lines of code to train the model. Its built-in model.fit() function handles the training process automatically, making it convenient for beginners and developers who want to build models quickly.



3. Why should we not interpret the two training-accuracy values as proof that one framework is superior?

We cannot say that one framework is better simply because it produces a higher training accuracy. The results can differ because of model architecture, random initialization, hyperparameters, and training conditions. Additionally, training accuracy measures performance on data the model has already seen rather than its ability to predict new data.



**Caveat:** Both numbers are in-sample training accuracy; they are not held-out test accuracy. A meaningful comparison needs controlled architectures, initialization, hyperparameters, evaluation data, and repeated experiments.